# Results paragraph: IQM vendor and preprocessing effects

Runs analyses to fill placeholders in the IQM results paragraph. Uses:
- **IQM columns**: `raw_neighbor_corr`, `t1post_neighbor_corr`, `raw_dwi_contrast`, `t1post_dwi_contrast`, `CNR0_median`–`CNR4_median`
- **Kruskal–Wallis** by scanner vendor for each IQM (p-values and η²)
- **Pairwise Wilcoxon** (rank-biserial r) for vendor comparisons
- **Paired Wilcoxon** (raw vs preprocessed) for NDC and dMRI contrast

**Figure 2 panel labels:** All references to "Fig. 2a", "Fig. 2b", etc. refer to the **final assembled figure** (Figure2.png / Figure2.ai), which is reordered from the panel layout in Figure2.ipynb. Mapping: **Fig. 2a** = CNR across shells (notebook panel C), **Fig. 2b** = b=0 tSNR (notebook panel D), **Fig. 2c** = NDC (notebook panel A), **Fig. 2d** = dMRI contrast (notebook panel B).

Output: filled paragraph with all XX placeholders replaced.

## Setup and data

In [31]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(purrr)
  library(arrow)
  library(fs)
  library(jsonlite)
})

config_candidates <- c(
  Sys.getenv("CONFIG_PATH", unset = ""),
  fs::path(".", "config.json"), fs::path("..", "config.json"), fs::path("..", "..", "config.json")
)
config_candidates <- normalizePath(unique(config_candidates[nzchar(config_candidates)]), winslash = "/", mustWork = FALSE)
config_path <- config_candidates[file.exists(config_candidates)][1]
if (is.na(config_path) || !nzchar(config_path)) stop("Could not locate config.json.")
config <- jsonlite::fromJSON(config_path)
project_root <- normalizePath(config$project_root, winslash = "/", mustWork = FALSE)

harm_path <- fs::path(project_root, "data", "harmonized_data", "merged_data_meisler_analyses_harmonized.parquet")
if (!file.exists(harm_path)) stop("Harmonized parquet not found: ", harm_path)

iqm_cols <- c(
  "raw_neighbor_corr", "t1post_neighbor_corr",
  "raw_dwi_contrast", "t1post_dwi_contrast",
  "CNR0_median", "CNR1_median", "CNR2_median", "CNR3_median", "CNR4_median"
)
df <- arrow::read_parquet(harm_path, col_select = c("scanner_manufacturer", iqm_cols)) %>%
  as_tibble() %>%
  filter(!is.na(scanner_manufacturer))

vendor_order <- c("GE", "Philips", "Siemens")
df <- df %>% mutate(scanner_manufacturer = factor(as.character(scanner_manufacturer), levels = vendor_order))
cat("N rows:", nrow(df), "\n")

N rows: 24187 


## Helper: Kruskal–Wallis η² and pairwise Wilcoxon rank-biserial r

In [32]:
# η² from Kruskal-Wallis: (H - k + 1)/(n - k); H = chi-squared stat, k = groups, n = N
kruskal_eta_sq <- function(x, g) {
  g <- factor(g)
  ok <- !is.na(x) & !is.na(g)
  x <- x[ok]; g <- g[ok]
  n <- length(x); k <- nlevels(g)
  if (n < 3L || k < 2L) return(NA_real_)
  ht <- kruskal.test(x ~ g)
  H <- unname(ht$statistic)
  (H - k + 1) / (n - k)
}

# Rank-biserial r: positive = first-named (ref) has LOWER values (second-named has higher values).
# R's wilcox.test(x1, x2) returns statistic = number of pairs with x1 > x2; use r = 1 - 2*stat/(n1*n2).
wilcox_r <- function(x, g, ref_level) {
  g <- factor(g)
  x1 <- x[g == ref_level]
  x2 <- x[g != ref_level]
  x1 <- x1[!is.na(x1)]; x2 <- x2[!is.na(x2)]
  n1 <- length(x1); n2 <- length(x2)
  if (n1 < 1L || n2 < 1L) return(NA_real_)
  wt <- wilcox.test(x1, x2, exact = FALSE)
  stat <- as.numeric(wt$statistic)  # pairs where x1 (ref) > x2
  r <- 1 - (2 * stat) / (n1 * n2)
  pmax(-1, pmin(1, r))
}

cat("Helpers defined.\n")

Helpers defined.


## Kruskal–Wallis by vendor (all IQMs)

In [33]:
kw_results <- tibble(
  iqm = iqm_cols,
  p_value = NA_real_,
  eta_sq = NA_real_
)

for (i in seq_along(iqm_cols)) {
  col <- iqm_cols[i]
  d <- df %>% filter(!is.na(.data[[col]]))
  if (nrow(d) < 10L) next
  ht <- kruskal.test(d[[col]] ~ d$scanner_manufacturer)
  kw_results$p_value[i] <- ht$p.value
  kw_results$eta_sq[i] <- kruskal_eta_sq(d[[col]], d$scanner_manufacturer)
}

kw_results <- kw_results %>%
  mutate(
    log10_p = log10(p_value),
    p_exponent = floor(log10_p)  # so p < 10^p_exponent
  )

# For paragraph: "p < 1×10-XX" => XX is positive (e.g. 15 for 1e-15); cap if p underflow
kw_min_exponent <- min(kw_results$p_exponent, na.rm = TRUE)
kw_paragraph_exponent <- -kw_min_exponent
if (!is.finite(kw_paragraph_exponent) || kw_paragraph_exponent > 300) kw_paragraph_exponent <- 300
kw_min_p <- min(kw_results$p_value, na.rm = TRUE)
cat("Kruskal-Wallis: min p =", format(kw_min_p, scientific = TRUE), "=> p < 1×10-", kw_paragraph_exponent, "\n")
print(kw_results)

Kruskal-Wallis: min p = 0e+00 => p < 1×10- 300 
# A tibble: 9 × 5
  iqm                  p_value eta_sq log10_p p_exponent
  <chr>                  <dbl>  <dbl>   <dbl>      <dbl>
1 raw_neighbor_corr          0  0.393    -Inf       -Inf
2 t1post_neighbor_corr       0  0.683    -Inf       -Inf
3 raw_dwi_contrast           0  0.169    -Inf       -Inf
4 t1post_dwi_contrast        0  0.146    -Inf       -Inf
5 CNR0_median                0  0.501    -Inf       -Inf
6 CNR1_median                0  0.593    -Inf       -Inf
7 CNR2_median                0  0.645    -Inf       -Inf
8 CNR3_median                0  0.666    -Inf       -Inf
9 CNR4_median                0  0.668    -Inf       -Inf


## η² for paragraph (Fig 2a–d)

In [34]:
# Fig 2a–d below refer to final Figure2.png/Figure2.ai (reordered from Figure2.ipynb panels A–D).
# Fig 2a: CNR across shells (CNR1–CNR4)
cnr_etas <- kw_results %>% filter(iqm %in% c("CNR1_median", "CNR2_median", "CNR3_median", "CNR4_median")) %>% pull(eta_sq)
eta_cnr_min <- round(min(cnr_etas, na.rm = TRUE), 2)
eta_cnr_max <- round(max(cnr_etas, na.rm = TRUE), 2)

# Fig 2b: b=0 tSNR (CNR0)
eta_tsnr <- kw_results %>% filter(iqm == "CNR0_median") %>% pull(eta_sq)
eta_tsnr <- round(eta_tsnr, 2)

# Fig 2c: NDC (raw and preprocessed; preprocessed = 0.68 given in paragraph)
eta_ndc_raw <- kw_results %>% filter(iqm == "raw_neighbor_corr") %>% pull(eta_sq)
eta_ndc_raw <- round(eta_ndc_raw, 2)

# Fig 2d: dMRI contrast (raw and preprocessed)
eta_contrast_raw <- kw_results %>% filter(iqm == "raw_dwi_contrast") %>% pull(eta_sq)
eta_contrast_pre <- kw_results %>% filter(iqm == "t1post_dwi_contrast") %>% pull(eta_sq)
eta_contrast_raw <- round(eta_contrast_raw, 2)
eta_contrast_pre <- round(eta_contrast_pre, 2)

cat("Fig 2a CNR η² range:", eta_cnr_min, "–", eta_cnr_max, "\n")
cat("Fig 2b tSNR η²:", eta_tsnr, "\n")
cat("Fig 2c NDC raw η²:", eta_ndc_raw, "\n")
cat("Fig 2d contrast raw η²:", eta_contrast_raw, "; preprocessed η²:", eta_contrast_pre, "\n")

Fig 2a CNR η² range: 0.59 – 0.67 
Fig 2b tSNR η²: 0.5 
Fig 2c NDC raw η²: 0.39 
Fig 2d contrast raw η²: 0.17 ; preprocessed η²: 0.15 


## Pairwise Wilcoxon (Siemens vs GE, Siemens vs Philips; rank-biserial r)

In [35]:
# Metrics where Siemens > GE and Siemens > Philips: CNR (all shells), tSNR (CNR0), NDC (raw + t1post)
metrics_siemens_higher <- c("CNR0_median", "CNR1_median", "CNR2_median", "CNR3_median", "CNR4_median",
                             "raw_neighbor_corr", "t1post_neighbor_corr")

pairwise_r <- expand_grid(
  iqm = metrics_siemens_higher,
  comparison = c("Siemens_vs_GE", "Siemens_vs_Philips")
) %>%
  mutate(r = NA_real_)

for (i in seq_len(nrow(pairwise_r))) {
  col <- pairwise_r$iqm[i]
  comp <- pairwise_r$comparison[i]
  other <- if (comp == "Siemens_vs_GE") "GE" else "Philips"
  d <- df %>% filter(scanner_manufacturer %in% c("Siemens", other), !is.na(.data[[col]]))
  pairwise_r$r[i] <- wilcox_r(d[[col]], d$scanner_manufacturer, ref_level = "Siemens")
}

# Paragraph: report min |r| so "r >= XX" is positive (r is negative when Siemens higher)
r_siemens_min <- min(abs(pairwise_r$r), na.rm = TRUE)
r_siemens_min_round <- round(r_siemens_min, 2)
cat("Siemens vs GE/Philips (CNR, tSNR, NDC): min |r| =", r_siemens_min_round, "\n")

# dMRI contrast: GE vs Siemens (small), Philips vs GE and Philips vs Siemens (large negative)
d_ge_siemens <- df %>% filter(scanner_manufacturer %in% c("GE", "Siemens"), !is.na(raw_dwi_contrast))
r_ge_siemens_raw <- wilcox_r(d_ge_siemens$raw_dwi_contrast, d_ge_siemens$scanner_manufacturer, ref_level = "Siemens")
r_ge_siemens_pre <- wilcox_r(d_ge_siemens$t1post_dwi_contrast, d_ge_siemens$scanner_manufacturer, ref_level = "Siemens")
r_ge_siemens <- max(abs(r_ge_siemens_raw), abs(r_ge_siemens_pre), na.rm = TRUE)
r_ge_siemens_round <- round(r_ge_siemens, 2)

# Philips vs GE/Siemens: ref_level = Philips so positive r = Philips lower (markedly lower contrast)
d_philips_ge <- df %>% filter(scanner_manufacturer %in% c("Philips", "GE"), !is.na(raw_dwi_contrast))
d_philips_siemens <- df %>% filter(scanner_manufacturer %in% c("Philips", "Siemens"), !is.na(raw_dwi_contrast))
r_philips_ge_raw <- wilcox_r(d_philips_ge$raw_dwi_contrast, d_philips_ge$scanner_manufacturer, ref_level = "Philips")
r_philips_siemens_raw <- wilcox_r(d_philips_siemens$raw_dwi_contrast, d_philips_siemens$scanner_manufacturer, ref_level = "Philips")
r_philips_ge_pre <- wilcox_r(d_philips_ge$t1post_dwi_contrast, d_philips_ge$scanner_manufacturer, ref_level = "Philips")
r_philips_siemens_pre <- wilcox_r(d_philips_siemens$t1post_dwi_contrast, d_philips_siemens$scanner_manufacturer, ref_level = "Philips")
r_philips_all <- c(r_philips_ge_raw, r_philips_siemens_raw, r_philips_ge_pre, r_philips_siemens_pre)
r_philips_min <- min(r_philips_all, na.rm = TRUE)
r_philips_max <- max(r_philips_all, na.rm = TRUE)
r_philips_range <- paste0(round(r_philips_min, 2), "–", round(r_philips_max, 2))
cat("GE vs Siemens contrast (small): max |r| =", r_ge_siemens_round, "\n")
cat("Philips vs GE/Siemens contrast: r range", r_philips_range, "\n")

Siemens vs GE/Philips (CNR, tSNR, NDC): min |r| = 0.73 
GE vs Siemens contrast (small): max |r| = 0.11 
Philips vs GE/Siemens contrast: r range 0.73–0.8 


### Table: Wilcoxon rank-biserial r by metric and vendor comparison

For each pair, **positive r = first-named vendor has lower values** (second-named has higher values). E.g. Philips vs Siemens with positive r means Philips has lower values (Siemens higher).

In [36]:
metric_labels <- c(
  raw_neighbor_corr = "NDC (raw)",
  t1post_neighbor_corr = "NDC (preprocessed)",
  raw_dwi_contrast = "dMRI contrast (raw)",
  t1post_dwi_contrast = "dMRI contrast (preprocessed)",
  CNR0_median = "b=0 tSNR (CNR0)",
  CNR1_median = "CNR b=500",
  CNR2_median = "CNR b=1000",
  CNR3_median = "CNR b=2000",
  CNR4_median = "CNR b=3000"
)

pairwise_spec <- expand_grid(
  iqm = iqm_cols,
  comparison = c("Siemens_vs_GE", "GE_vs_Philips", "Philips_vs_Siemens")
)

pairwise_spec <- pairwise_spec %>%
  mutate(
    ref = case_when(
      comparison == "Siemens_vs_GE" ~ "Siemens",
      comparison == "GE_vs_Philips" ~ "GE",
      comparison == "Philips_vs_Siemens" ~ "Philips",
      TRUE ~ NA_character_
    ),
    other = case_when(
      comparison == "Siemens_vs_GE" ~ "GE",
      comparison == "GE_vs_Philips" ~ "Philips",
      comparison == "Philips_vs_Siemens" ~ "Siemens",
      TRUE ~ NA_character_
    ),
    r = NA_real_
  )

for (i in seq_len(nrow(pairwise_spec))) {
  col <- pairwise_spec$iqm[i]
  ref <- pairwise_spec$ref[i]
  oth <- pairwise_spec$other[i]
  d <- df %>% filter(scanner_manufacturer %in% c(ref, oth), !is.na(.data[[col]]))
  pairwise_spec$r[i] <- wilcox_r(d[[col]], d$scanner_manufacturer, ref_level = ref)
}

pairwise_table_long <- pairwise_spec %>%
  mutate(metric = unname(metric_labels[iqm])) %>%
  select(metric, comparison, r)

pairwise_table_wide <- pairwise_table_long %>%
  mutate(r = round(r, 3)) %>%
  pivot_wider(names_from = comparison, values_from = r) %>%
  rename(
    `Siemens vs GE` = Siemens_vs_GE,
    `GE vs Philips` = GE_vs_Philips,
    `Philips vs Siemens` = Philips_vs_Siemens
  )

cat("Wilcoxon rank-biserial r: positive = first-named vendor has LOWER values (second-named higher)\n\n")
print(pairwise_table_wide)

Wilcoxon rank-biserial r: positive = first-named vendor has LOWER values (second-named higher)

# A tibble: 9 × 4
  metric                    `Siemens vs GE` `GE vs Philips` `Philips vs Siemens`
  <chr>                               <dbl>           <dbl>                <dbl>
1 NDC (raw)                          -0.734          -0.045                0.811
2 NDC (preprocessed)                 -0.99            0.63                 0.976
3 dMRI contrast (raw)                -0.11           -0.794                0.8  
4 dMRI contrast (preproces…           0.074          -0.783                0.734
5 b=0 tSNR (CNR0)                    -0.848          -0.146                0.862
6 CNR b=500                          -0.936          -0.29                 0.897
7 CNR b=1000                         -0.978          -0.254                0.935
8 CNR b=2000                         -0.993          -0.216                0.95 
9 CNR b=3000                         -0.995           0.055                0

## Paired Wilcoxon (raw vs preprocessed: NDC and dMRI contrast)

In [37]:
df_paired <- df %>% filter(!is.na(raw_neighbor_corr) & !is.na(t1post_neighbor_corr) & !is.na(raw_dwi_contrast) & !is.na(t1post_dwi_contrast))
paired_ndc <- wilcox.test(df_paired$raw_neighbor_corr, df_paired$t1post_neighbor_corr, paired = TRUE, exact = FALSE)
paired_contrast <- wilcox.test(df_paired$raw_dwi_contrast, df_paired$t1post_dwi_contrast, paired = TRUE, exact = FALSE)
p_paired_ndc <- paired_ndc$p.value
p_paired_contrast <- paired_contrast$p.value
cat("Paired Wilcoxon NDC (raw vs t1post): p =", format(p_paired_ndc, digits = 3), "\n")
cat("Paired Wilcoxon dMRI contrast (raw vs t1post): p =", format(p_paired_contrast, digits = 3), "\n")
cat("Both p < 0.001:", p_paired_ndc < 0.001 & p_paired_contrast < 0.001, "\n")

Paired Wilcoxon NDC (raw vs t1post): p = 0 
Paired Wilcoxon dMRI contrast (raw vs t1post): p = 0 
Both p < 0.001: TRUE 


### Effect size table: paired Wilcoxon (raw vs preprocessed)

Rank-biserial r = Z / sqrt(N); **positive r = preprocessed > raw** (within-scan improvement).

In [ ]:
# Paired Wilcoxon: r = Z / sqrt(N); Z from signed-rank statistic V (R returns V = sum of ranks of positive diffs).
wilcox_paired_r <- function(x_raw, x_pre) {
  d <- x_pre - x_raw
  ok <- !is.na(d) & (d != 0)
  n <- sum(ok)
  if (n < 2L) return(list(r = NA_real_, p = NA_real_, n_pairs = n))
  wt <- wilcox.test(x_pre, x_raw, paired = TRUE, exact = FALSE)
  V <- as.numeric(wt$statistic)
  # Z = (V - E(V)) / sd(V); E(V) = n*(n+1)/4, Var(V) = n*(n+1)*(2*n+1)/24
  mu <- n * (n + 1) / 4
  sig <- sqrt(n * (n + 1) * (2 * n + 1) / 24)
  Z <- (V - mu) / sig
  r <- Z / sqrt(n)
  r <- pmax(-1, pmin(1, r))
  list(r = r, p = wt$p.value, n_pairs = n)
}

paired_ndc_es <- wilcox_paired_r(df_paired$raw_neighbor_corr, df_paired$t1post_neighbor_corr)
paired_contrast_es <- wilcox_paired_r(df_paired$raw_dwi_contrast, df_paired$t1post_dwi_contrast)

paired_effect_table <- tibble(
  metric = c("NDC (neighboring dMRI correlation)", "dMRI contrast"),
  n_pairs = c(paired_ndc_es$n_pairs, paired_contrast_es$n_pairs),
  r = c(paired_ndc_es$r, paired_contrast_es$r),
  p_value = c(paired_ndc_es$p, paired_contrast_es$p)
) %>%
  mutate(
    r = round(r, 3),
    p_value = format(p_value, scientific = TRUE, digits = 2)
  )

cat("Paired Wilcoxon signed-rank: positive r = preprocessed > raw (improvement)\n\n")
print(paired_effect_table)

## Filled results paragraph

In [38]:
txt <- sprintf(
  paste(
    "Substantial differences in IQMs were observed between scanner vendors in both raw and preprocessed data (all Kruskal-Wallis tests p < 1e%d).",
    "Vendor effects were large for CNR across all diffusion-weighted shells (Fig. 2a; eta2 = %s-%s), b = 0 tSNR (Fig. 2b; eta2 = %s), and NDC (Fig. 2c; raw eta2 = %s; preprocessed eta2 = 0.68).",
    "Vendor differences in dMRI contrast were more modest by contrast (Fig. 2d; raw eta2 = %s; preprocessed eta2 = %s).",
    "Pairwise comparisons revealed systematic vendor-specific patterns. Siemens scanners exhibited higher CNR, tSNR, and NDC values than both GE and Philips scanners (all pairwise Wilcoxon rank-biserial r >= %s).",
    "Differences between GE and Siemens scanners were generally small for dMRI contrast (r <= %s), whereas Philips scanners showed markedly lower dMRI contrast than both GE and Siemens scanners (rank-biserial r = %s).",
    "Preprocessing produced highly consistent within-scan improvements in both NDC and dMRI contrast (Fig. 2c and 2d; paired Wilcoxon signed-rank tests both p < 0.001).",
    "Additional characterization of other IQMs, as well as IQMs stratified by acquisition site, are provided in Supplementary Fig. S2.",
    "In sum, preprocessing demonstrably improved image quality, yet there were still robust quality differences across scanner vendor, underscoring scanner heterogeneity as a fundamental challenge for large multisite diffusion MRI studies.",
    sep = " "
  ),
  kw_paragraph_exponent, eta_cnr_min, eta_cnr_max, eta_tsnr, eta_ndc_raw,
  eta_contrast_raw, eta_contrast_pre,
  r_siemens_min_round, r_ge_siemens_round, r_philips_range
)
cat(txt, "\n")

Substantial differences in IQMs were observed between scanner vendors in both raw and preprocessed data (all Kruskal-Wallis tests p < 1e300). Vendor effects were large for CNR across all diffusion-weighted shells (Fig. 2a; eta2 = 0.59-0.67), b = 0 tSNR (Fig. 2b; eta2 = 0.5), and NDC (Fig. 2c; raw eta2 = 0.39; preprocessed eta2 = 0.68). Vendor differences in dMRI contrast were more modest by contrast (Fig. 2d; raw eta2 = 0.17; preprocessed eta2 = 0.15). Pairwise comparisons revealed systematic vendor-specific patterns. Siemens scanners exhibited higher CNR, tSNR, and NDC values than both GE and Philips scanners (all pairwise Wilcoxon rank-biserial r >= 0.73). Differences between GE and Siemens scanners were generally small for dMRI contrast (r <= 0.11), whereas Philips scanners showed markedly lower dMRI contrast than both GE and Siemens scanners (rank-biserial r = 0.73–0.8). Preprocessing produced highly consistent within-scan improvements in both NDC and dMRI contrast (Fig. 2c and 2d;